# 🎬 Reels — Free-GPU Talking Avatar (Kaggle / MuseTalk)

Enable **Settings → Accelerator → GPU T4 x2** and **Internet: On**, then run top to bottom.

Output: a 9:16 `reel.mp4` you download from the Kaggle output panel.


In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch;print('CUDA:',torch.cuda.is_available())

In [ ]:
# 2) Get repo
%cd /kaggle/working
!git clone https://github.com/amsinghnavdeep/reels.git 2>/dev/null || (cd reels && git pull)
%cd /kaggle/working/reels
!pip -q install edge-tts pydub pyyaml

In [ ]:
# 3) Clone + install MuseTalk (one-time)
import os
os.makedirs('engines',exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk
%cd /kaggle/working/reels/engines/MuseTalk
!pip -q install -r requirements.txt
!pip -q install --no-cache-dir -U openmim
!mim install mmengine 'mmcv==2.0.1' 'mmdet==3.1.0' 'mmpose==1.1.0'
!mkdir -p ffmpeg && cd ffmpeg && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz --strip-components=1
os.environ['FFMPEG_PATH']='/kaggle/working/reels/engines/MuseTalk/ffmpeg'

In [ ]:
# 4) Download weights (~5 GB).
#    MuseTalk's download_weights.sh uses the removed `huggingface-cli` (downloads
#    nothing); use the huggingface_hub Python API instead.
%cd /kaggle/working/reels/engines/MuseTalk
!curl -sL https://raw.githubusercontent.com/amsinghnavdeep/reels/main/notebooks/dl_weights.py -o /tmp/dl.py && python3 /tmp/dl.py

In [ ]:
# 5) Driving VIDEO + cloned voice. Add your files via 'Add Data' or upload to /kaggle/working,
#    then set DRIVE_VIDEO / VOICE_SRC to their paths. Real footage = sharp mouth + real motion.
%cd /kaggle/working/reels
import os, subprocess
os.makedirs('output', exist_ok=True)
DRIVE_VIDEO = '/kaggle/input/YOUR-DATASET/your_talking_head.mp4'   # <-- set this
VOICE_SRC   = '/kaggle/input/YOUR-DATASET/your_voice.m4a'          # <-- set this
SCRIPT = 'examples/script.txt'   # edit with your words

# Optional: trim/crop out burned captions. e.g. TRIM='-ss 15 -to 19.5'  CROP='crop=360:360:0:48'
TRIM = ''
CROP = ''
if TRIM or CROP:
    vf = f'-vf "{CROP}"' if CROP else ''
    subprocess.run(f'ffmpeg -y {TRIM} -i "{DRIVE_VIDEO}" {vf} -an -r 25 -c:v libx264 -pix_fmt yuv420p output/drive_clip.mp4', shell=True, check=True)
    DRIVE_VIDEO = 'output/drive_clip.mp4'
print('driving video:', DRIVE_VIDEO)
subprocess.run(f'ffmpeg -y -i "{VOICE_SRC}" -ar 16000 -ac 1 output/voice_ref.wav', shell=True, check=True)
!pip -q install coqui-tts
!COQUI_TOS_AGREED=1 python tts.py --script $SCRIPT --engine xtts --clone output/voice_ref.wav --lang hi --output output/speech.wav
#   Or a clean non-cloned Hindi voice:  !python tts.py --script $SCRIPT --engine edge --voice hi-IN-MadhurNeural --output output/speech.wav
from IPython.display import Audio; Audio('output/speech.wav')

In [ ]:
# 6) MuseTalk lip-sync
import yaml,os
os.chdir('/kaggle/working/reels/engines/MuseTalk')
cfg={'task_0':{'video_path':('/kaggle/working/reels/'+DRIVE_VIDEO) if not DRIVE_VIDEO.startswith('/') else DRIVE_VIDEO,'audio_path':'/kaggle/working/reels/output/speech.wav'}}
os.makedirs('configs/inference',exist_ok=True)
open('configs/inference/reels.yaml','w').write(yaml.dump(cfg))
# guard MuseTalk's unconditional temp cleanup crash on still-image avatar input
!sed -i 's|^            shutil.rmtree(save_dir_full)|            if get_file_type(video_path) == "video": shutil.rmtree(save_dir_full)|' scripts/inference.py
!python -m scripts.inference --inference_config configs/inference/reels.yaml --result_dir /kaggle/working/reels/output/muse --unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json --version v15 --fps 25 --batch_size 4 --use_float16 --parsing_mode jaw --extra_margin 10

In [ ]:
# 7) 9:16 reel
%cd /kaggle/working/reels
import glob,os
clips=sorted(glob.glob('output/muse/**/*.mp4',recursive=True),key=os.path.getmtime)
assert clips,'No MuseTalk output — check previous cell logs.'
raw=clips[-1];print('raw:',raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
print('Download output/reel.mp4 from the Kaggle Output panel.')
from IPython.display import Video; Video('output/reel.mp4',embed=True,width=270)